In [8]:
import sympy as sp

def analizar_hartman_grobman():
    # Definir las variables simbólicas
    G, I, B = sp.symbols('G I B', real=True)

    # 2. Definir las constantes usando los valores de la tabla
    R_0 = 864
    G_e = 140
    E_GO = 1.44
    S_I = 0.72
    sigma = 43.2
    alpha = 20000
    rho = 0.41
    k = 432
    d_0 = 0.06
    r_1 = 0.84e-3   # Equivale a 0.84 * 10^-3
    r_2 = 0.24e-5   # Equivale a 0.24 * 10^-5

    
    # Definir el sistema de ecuaciones diferenciales no lineales
    # Ejemplo: x' = y, y' = x - x^3 - y
    f1 = R_0 + G_e - (E_GO + S_I * I) * G
    f2 = (B * sigma * G**2) / (alpha + G**2) - (rho + k) * I
    f3 = (-d_0 + r_1 * G - r_2 * G**2) * B
    
    print("SISTEMA DE ECUACIONES:")
    print(f"dG/dt = {f1}")
    print(f"dI/dt = {f2}")
    print(f"dB/dt = {f3}")
    print("-" * 50)
    
    # --- PASO 1: Encontrar los puntos críticos ---
    print("\n[PASO 1] Encontrando puntos críticos (f(x,y) = 0)...")
    puntos_criticos = sp.solve([f1, f2, f3], [G, I, B], dict=True)
    print(f"Se encontraron {len(puntos_criticos)} puntos críticos:")
    for i, pt in enumerate(puntos_criticos):
        print(f"  Punto {i+1}: ({pt[G]}, {pt[I]}, {pt[B]})")
        
    # --- PASO 2: Calcular la Matriz Jacobiana general ---
    print("\n[PASO 2] Calculando la Matriz Jacobiana simbólica...")
    # Creamos una matriz columna con nuestras funciones y aplicamos el jacobiano
    sistema_matriz = sp.Matrix([f1, f2, f3])
    J = sistema_matriz.jacobian([G, I, B])
    sp.pprint(J)
    
    # Iterar sobre cada punto crítico encontrado
    for i, pt in enumerate(puntos_criticos):
        print("=" * 50)
        print(f"ANALIZANDO EL PUNTO {i+1}: ({pt[G]}, {pt[I]}, {pt[B]})")
        
        # Evaluar la Jacobiana en el punto crítico
        J_eval = J.subs({G: pt[G], I: pt[I], B: pt[B]})
        print("\nMatriz Jacobiana evaluada en el punto:")
        sp.pprint(J_eval)
        
        # --- PASO 3: Calcular los valores propios ---
        print("\n[PASO 3] Calculando Valores Propios (Eigenvalores)...")
        # eigenvals() devuelve un diccionario {eigenvalor: multiplicidad}
        eigenvalores = J_eval.eigenvals() 
        
        partes_reales = []
        partes_imaginarias = []
        
        for eigenval, multiplicidad in eigenvalores.items():
            # Simplificar y separar parte real e imaginaria
            eval_complejo = sp.N(eigenval)
            real = sp.re(eval_complejo)
            imag = sp.im(eval_complejo)
            
            partes_reales.append(real)
            partes_imaginarias.append(imag)
            
            print(f"  λ = {eval_complejo}")
            
        # --- PASO 4: Verificar hiperbolicidad ---
        print("\n[PASO 4] Verificando condición de hiperbolicidad...")
        # Un punto es hiperbólico si NINGUNA parte real es cero
        es_hiperbolico = all(abs(r) > 1e-9 for r in partes_reales)
        
        if not es_hiperbolico:
            print("  -> El punto NO es hiperbólico (hay un valor propio con parte real 0).")
            print("  -> EL TEOREMA DE HARTMAN-GROBMAN NO ES CONCLUYENTE AQUÍ.")
            continue
        else:
            print("  -> El punto ES hiperbólico. Podemos proceder a clasificar.")
            
        # --- PASO 5: Clasificación del punto ---
        print("\n[PASO 5] Clasificación según el Teorema...")
        
        tiene_imaginarios = any(abs(i) > 1e-9 for i in partes_imaginarias)
        todos_negativos = all(r < -1e-9 for r in partes_reales)
        todos_positivos = all(r > 1e-9 for r in partes_reales)
        
        if todos_negativos:
            if tiene_imaginarios:
                print("  => RESULTADO: Foco Estable (Espiral atractor)")
            else:
                print("  => RESULTADO: Nodo Estable (Sumidero atractor)")
                
        elif todos_positivos:
            if tiene_imaginarios:
                print("  => RESULTADO: Foco Inestable (Espiral repulsor)")
            else:
                print("  => RESULTADO: Nodo Inestable (Fuente repulsora)")
                
        else:
            # Mezcla de positivos y negativos
            print("  => RESULTADO: Punto de Silla (Saddle)")

analizar_hartman_grobman()

SISTEMA DE ECUACIONES:
dG/dt = -G*(0.72*I + 1.44) + 1004
dI/dt = 43.2*B*G**2/(G**2 + 20000) - 432.41*I
dB/dt = B*(-2.4e-6*G**2 + 0.00084*G - 0.06)
--------------------------------------------------

[PASO 1] Encontrando puntos críticos (f(x,y) = 0)...
Se encontraron 3 puntos críticos:
  Punto 1: (697.222222222222, 0.0, 0.0)
  Punto 2: (250.000000000000, 3.57777777777778, 47.2714882716049)
  Punto 3: (100.000000000000, 11.9444444444444, 358.673418209877)

[PASO 2] Calculando la Matriz Jacobiana simbólica...
⎡       -0.72⋅I - 1.44         -0.72⋅G                0               ⎤
⎢                                                                     ⎥
⎢            3                                           2            ⎥
⎢    86.4⋅B⋅G       86.4⋅B⋅G                       43.2⋅G             ⎥
⎢- ───────────── + ──────────  -432.41            ──────────          ⎥
⎢              2    2                              2                  ⎥
⎢  ⎛ 2        ⎞    G  + 20000                     G  + 2

In [ ]:
import sympy as sp
#no
def main():
    # 1. Definir las variables de estado (Incógnitas)
    G, I, beta = sp.symbols('G I beta', real=True)

    # 2. Definir los parámetros del modelo
    R_0, G_e, E_G0, S_I = sp.symbols('R_0 G_e E_G0 S_I', real=True)
    sigma, alpha, rho, k = sp.symbols('sigma alpha rho k', real=True)
    d_0, r_1, r_2 = sp.symbols('d_0 r_1 r_2', real=True)

    # 3. Construir las ecuaciones (Lado derecho de las EDOs)
    eq_G = R_0 + G_e - (E_G0 + S_I * I) * G
    eq_I = (beta * sigma * G**2) / (alpha + G**2) - (rho + k) * I
    eq_beta = (-d_0 + r_1 * G - r_2 * G**2) * beta

    print("--- CALCULANDO PUNTOS CRÍTICOS SIMBÓLICOS ---")
    # Igualamos a cero y resolvemos el sistema para G, I y beta
    puntos_criticos = sp.solve([eq_G, eq_I, eq_beta], (G, I, beta))

    # Mostrar las soluciones simbólicas encontradas
    for i, pt in enumerate(puntos_criticos):
        print(f"\nPunto crítico {i + 1}:")
        print(f"G* = {pt[0]}")
        print(f"I* = {pt[1]}")
        print(f"beta* = {pt[2]}")

    print("\n" + "="*50 + "\n")
    print("--- SUSTITUYENDO VALORES NUMÉRICOS ---")
    
    # 4. Diccionario de parámetros con valores de ejemplo
    # (Sustituye estos números por tus datos experimentales o de la literatura)
    valores_parametros = {
        R_0: 864,    # Ejemplo: Tasa de producción de glucosa
        G_e: 140,      # Ejemplo: Glucosa exógena
        E_G0: 1.44,  # Ejemplo: Efectividad de la glucosa
        S_I: 0.72,   # Ejemplo: Sensibilidad a la insulina
        sigma: 43.2, # Ejemplo: Tasa máxima de secreción de insulina
        alpha: 20000,# Ejemplo: Constante de Michaelis-Menten
        rho: 0.41,   # Ejemplo: Degradación de insulina
        k: 432,        # Ejemplo: Aclaramiento
        d_0: 0.06,   # Ejemplo: Tasa de muerte natural de células beta
        r_1: 0.00084, # Ejemplo: Tasa de replicación dependiente de glucosa
        r_2: 0.0000024# Ejemplo: Toxicidad por glucosa
    }

    # Sustituimos los parámetros en las soluciones simbólicas obtenidas
    for i, pt in enumerate(puntos_criticos):
        print(f"\nEvaluando Punto Crítico {i + 1}:")
        
        # Obtenemos los valores sustituyendo los parámetros
        G_val = pt[0].subs(valores_parametros)
        I_val = pt[1].subs(valores_parametros)
        beta_val = pt[2].subs(valores_parametros)
        
        # Si la solución matemática genera números complejos o raíces,
        # evalf() lo calcula numéricamente a decimales.
        try:
            print(f"G* ≈ {G_val.evalf(5)}")
            print(f"I* ≈ {I_val.evalf(5)}")
            print(f"beta* ≈ {beta_val.evalf(5)}")
        except Exception as e:
            print(f"No se pudo evaluar completamente (posiblemente fuera del dominio real): {e}")

if __name__ == "__main__":
    main()

--- CALCULANDO PUNTOS CRÍTICOS SIMBÓLICOS ---


KeyboardInterrupt: 

In [13]:
import sympy as sp

def main():
    # 1. Definir variables
    G, I, beta = sp.symbols('G I beta', real=True)

    # 2. Definir parámetros
    R_0, G_e, E_G0, S_I = sp.symbols('R_0 G_e E_G0 S_I', real=True)
    sigma, alpha, rho, k = sp.symbols('sigma alpha rho k', real=True)
    d_0, r_1, r_2 = sp.symbols('d_0 r_1 r_2', real=True)

    print("--- CALCULANDO PUNTOS CRÍTICOS SIMBÓLICOS (PASO A PASO) ---")
    puntos_criticos = []

    # CASO 1: Falla de células beta (beta = 0)
    # Si beta = 0, entonces de la ec. 2, I = 0.
    # De la ec. 1: R_0 + G_e - E_G0*G = 0 -> G = (R_0 + G_e) / E_G0
    G_falla = (R_0 + G_e) / E_G0
    I_falla = 0
    beta_falla = 0
    puntos_criticos.append((G_falla, I_falla, beta_falla))

    # CASO 2: Células beta funcionantes (beta != 0)
    # De la ec. 3: -d_0 + r_1*G - r_2*G**2 = 0
    ec_cuadratica_G = -d_0 + r_1 * G - r_2 * G**2
    G_raices = sp.solve(ec_cuadratica_G, G)

    for G_sol in G_raices:
        # De la ec. 1: Despejamos I en términos de G
        # R_0 + G_e - (E_G0 + S_I*I)*G = 0
        I_sol = (R_0 + G_e - E_G0 * G_sol) / (S_I * G_sol)
        
        # De la ec. 2: Despejamos beta en términos de I y G
        # (beta * sigma * G**2) / (alpha + G**2) - (rho + k)*I = 0
        beta_sol = (rho + k) * I_sol * (alpha + G_sol**2) / (sigma * G_sol**2)
        
        puntos_criticos.append((G_sol, I_sol, beta_sol))

    # Mostrar las soluciones simbólicas
    for i, pt in enumerate(puntos_criticos):
        print(f"\nPunto crítico {i + 1}:")
        print(f"G* = {pt[0]}")
        print(f"I* = {pt[1]}")
        print(f"beta* = {pt[2]}")

    print("\n" + "="*50 + "\n")
    print("--- SUSTITUYENDO VALORES NUMÉRICOS ---")
    
    # 4. Diccionario de parámetros
    valores_parametros = {
        R_0: 864, G_e: 3, E_G0: 1.44, S_I: 0.72,
        sigma: 43.2, alpha: 20000, rho: 4.32, k: 3,
        d_0: 0.06, r_1: 0.0014, r_2: 0.000007
    }

    # Sustituimos los parámetros
    for i, pt in enumerate(puntos_criticos):
        print(f"\nEvaluando Punto Crítico {i + 1}:")
        try:
            G_val = pt[0].subs(valores_parametros).evalf(5)
            I_val = pt[1].subs(valores_parametros).evalf(5)
            beta_val = pt[2].subs(valores_parametros).evalf(5)
            
            print(f"G* ≈ {G_val}")
            print(f"I* ≈ {I_val}")
            print(f"beta* ≈ {beta_val}")
            
            # Pequeña validación para descartar valores biológicamente imposibles (negativos)
            if G_val < 0 or I_val < 0 or beta_val < 0:
                print("  (Nota: Este punto tiene valores negativos, por lo que podría no tener sentido biológico).")
                
        except Exception as e:
            print(f"No se pudo evaluar: {e}")

if __name__ == "__main__":
    main()

--- CALCULANDO PUNTOS CRÍTICOS SIMBÓLICOS (PASO A PASO) ---

Punto crítico 1:
G* = (G_e + R_0)/E_G0
I* = 0
beta* = 0

Punto crítico 2:
G* = (r_1 - sqrt(-4*d_0*r_2 + r_1**2))/(2*r_2)
I* = 2*r_2*(-E_G0*(r_1 - sqrt(-4*d_0*r_2 + r_1**2))/(2*r_2) + G_e + R_0)/(S_I*(r_1 - sqrt(-4*d_0*r_2 + r_1**2)))
beta* = 8*r_2**3*(alpha + (r_1 - sqrt(-4*d_0*r_2 + r_1**2))**2/(4*r_2**2))*(k + rho)*(-E_G0*(r_1 - sqrt(-4*d_0*r_2 + r_1**2))/(2*r_2) + G_e + R_0)/(S_I*sigma*(r_1 - sqrt(-4*d_0*r_2 + r_1**2))**3)

Punto crítico 3:
G* = (r_1 + sqrt(-4*d_0*r_2 + r_1**2))/(2*r_2)
I* = 2*r_2*(-E_G0*(r_1 + sqrt(-4*d_0*r_2 + r_1**2))/(2*r_2) + G_e + R_0)/(S_I*(r_1 + sqrt(-4*d_0*r_2 + r_1**2)))
beta* = 8*r_2**3*(alpha + (r_1 + sqrt(-4*d_0*r_2 + r_1**2))**2/(4*r_2**2))*(k + rho)*(-E_G0*(r_1 + sqrt(-4*d_0*r_2 + r_1**2))/(2*r_2) + G_e + R_0)/(S_I*sigma*(r_1 + sqrt(-4*d_0*r_2 + r_1**2))**3)


--- SUSTITUYENDO VALORES NUMÉRICOS ---

Evaluando Punto Crítico 1:
No se pudo evaluar: 'int' object has no attribute 'subs'

Evaluand

In [2]:
import sympy as sp

def clasificar_puntos_criticos(ecuaciones, variables, puntos_criticos):
    """
    Analiza y clasifica una lista de puntos críticos de un sistema dinámico 
    usando el método de Hartman-Grobman.
    """
    print("=== ANÁLISIS DE PUNTOS CRÍTICOS (HARTMAN-GROBMAN) ===\n")
    
    # Paso 2: Obtener la Matriz Jacobiana de forma simbólica
    sistema_matriz = sp.Matrix(ecuaciones)
    J = sistema_matriz.jacobian(variables)
    
    print("1. Matriz Jacobiana General:")
    sp.pprint(J)
    print("\n" + "="*50 + "\n")
    
    # Iterar sobre cada punto crítico dado
    for i, punto in enumerate(puntos_criticos, 1):
        print(f"--- Evaluando Punto Crítico {i}: {punto} ---")
        
        # Crear un diccionario para sustituir las variables por los valores del punto
        subs_dict = dict(zip(variables, punto))
        
        # Evaluar la matriz Jacobiana en el punto crítico
        J_eval = J.subs(subs_dict)
        print("Matriz Jacobiana evaluada:")
        sp.pprint(J_eval)
        
        # Paso 3: Calcular los eigenvalores (valores propios)
        # eigenvals() devuelve un diccionario {eigenvalor: multiplicidad}
        eigenvalores_dict = J_eval.eigenvals()
        eigenvalores = list(eigenvalores_dict.keys())
        
        print("\nEigenvalores:")
        for ev in eigenvalores:
            print(f"  λ = {ev}")
        
        # Extraer las partes reales e imaginarias
        partes_reales = [sp.re(ev) for ev in eigenvalores]
        partes_imag = [sp.im(ev) for ev in eigenvalores]
        
        # Paso 4: Verificar hiperbolicidad
        # Es hiperbólico si NINGUNA parte real es exactamente cero
        es_hiperbolico = all(r != 0 for r in partes_reales)
        
        # Paso 5: Clasificación
        print("\nDiagnóstico:")
        if not es_hiperbolico:
            print("  ⚠️ NO HIPERBÓLICO.")
            print("  Al menos un eigenvalor tiene parte real cero.")
            print("  El Teorema de Hartman-Grobman NO ES CONCLUYENTE aquí.")
        else:
            print("  ✓ Punto Hiperbólico. Procediendo a clasificar...")
            
            tiene_imaginarios = any(im != 0 for im in partes_imag)
            
            if all(r < 0 for r in partes_reales):
                if tiene_imaginarios:
                    print("  CLASIFICACIÓN: Foco / Espiral Atractor (Sumidero Estable)")
                else:
                    print("  CLASIFICACIÓN: Nodo Atractor (Sumidero Estable)")
                    
            elif all(r > 0 for r in partes_reales):
                if tiene_imaginarios:
                    print("  CLASIFICACIÓN: Foco / Espiral Repulsor (Fuente Inestable)")
                else:
                    print("  CLASIFICACIÓN: Nodo Repulsor (Fuente Inestable)")
                    
            else:
                print("  CLASIFICACIÓN: Punto de Silla (Saddle Inestable)")
                
        print("="*50 + "\n")

# ==========================================
# EJEMPLO DE USO
# ==========================================
if __name__ == "__main__":
    # 1. Definir las variables simbólicas (G, I, B donde B representa las células beta)
    G, I, B = sp.symbols('G I B', real=True)

    # 2. Definir las constantes usando los valores de la tabla
    R_0 = 864
    G_e = 140
    E_GO = 1.44
    S_I = 0.72
    sigma = 43.2
    alpha = 20000
    rho = 0.41
    k = 432
    d_0 = 0.06
    r_1 = 0.84e-3   # Equivale a 0.84 * 10^-3
    r_2 = 0.24e-5   # Equivale a 0.24 * 10^-5

    variables = [G, I, B]
    
    # Definir el sistema de ecuaciones no lineales
    # Ejemplo: dx/dt = y, dy/dt = x - x^3 - y
    ecuacion_1 = R_0 + G_e - (E_GO + S_I * I) * G
    ecuacion_2 = (B * sigma * G*2) / (alpha + G*2) - (rho + k) * I
    ecuacion_3 = (-d_0 + r_1 * G - r_2 * G**2) * B
    sistema = [ecuacion_1, ecuacion_2, ecuacion_3]
    
    # Definir los puntos críticos que ya conoces
    # En este sistema, al igualar a cero, los puntos son (-1, 0, 0), (0, 0, 0) y (1, 0, 0)
    puntos = [(697.22, 0, 0), (0, 0, 0), (1, 0, 0)]
    
    # Ejecutar la función
    clasificar_puntos_criticos(sistema, variables, puntos)

=== ANÁLISIS DE PUNTOS CRÍTICOS (HARTMAN-GROBMAN) ===

1. Matriz Jacobiana General:
⎡        -0.72⋅I - 1.44          -0.72⋅G                0               ⎤
⎢                                                                       ⎥
⎢    172.8⋅B⋅G        86.4⋅B                         86.4⋅G             ⎥
⎢- ────────────── + ───────────  -432.41           ───────────          ⎥
⎢               2   2⋅G + 20000                    2⋅G + 20000          ⎥
⎢  (2⋅G + 20000)                                                        ⎥
⎢                                                                       ⎥
⎢                                                   2                   ⎥
⎣    B⋅(0.00084 - 4.8e-6⋅G)         0     - 2.4e-6⋅G  + 0.00084⋅G - 0.06⎦


--- Evaluando Punto Crítico 1: (697.22, 0, 0) ---
Matriz Jacobiana evaluada:
⎡-1.44  -501.9984         0        ⎤
⎢                                  ⎥
⎢  0     -432.41   2.81567584849148⎥
⎢                                  ⎥
⎣  0        0       -0.